# Running the experiments

This notebook reruns the three experiments of the paper *Does Better Retrieval Mean Better Answers?* on a Google Colab GPU. It fetches the code from the project repository, keeps every result on your Google Drive, and continues where it stopped if Colab disconnects (on the full run it will, more than once).

Run the cells in order. The text above each cell says what it does and what you should see when it has finished.

| Section | What happens | Time on a T4, pilot / full |
|---|---|---|
| 1 to 4 | Setup | about 5 minutes |
| 5 | Self-test on toy data | 1 minute |
| 6 | Experiment 1: retrieval on TREC-COVID | 20 minutes / 1 to 1.5 hours |
| 7 | Experiment 3: passage length and number of passages | 15 minutes / 1 to 2 hours |
| 8 | Experiment 2: question answering, seven conditions | 30 minutes / 3 to 5 hours per reader |
| 9 and 10 | Tables, figures, download | a few minutes |

The times are rough. They depend on how busy Colab is, and the first run is slower because the models and data have to be downloaded.

If you only want to check the numbers in the paper, you don't need this notebook or a GPU. `02_analysis_and_figures.ipynb` rebuilds all tables and figures from the saved results.

**When Colab disconnects.** The running cell stops, even if the page says "Resuming execution". Your work is not lost: embeddings are saved every 10,000 documents and answers every 64 questions. Click *Reconnect*, run sections 1 to 4 again, look at *Check progress*, and then rerun the cell that was interrupted. It picks up from the last save.

## 1. GPU and Hugging Face login

Before running the cell, choose *Runtime > Change runtime type > T4 GPU* and save. The output should report a Tesla T4 with about 14.5 GiB free. If much less is free, choose *Runtime > Restart session* and run the cell again.

You need a Hugging Face token for the second reader, Llama-3.1-8B-Instruct, which is a gated model. Once, before the first run:

1. Log in at huggingface.co, open the [Llama-3.1-8B-Instruct page](https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct) and accept the licence. Approval usually comes by e-mail within an hour.
2. Create a token under *Settings > Access Tokens* (a *Read* token is enough).
3. In Colab, click the key icon in the left sidebar, add a secret called `HF_TOKEN` with the token as its value, and switch on *Notebook access*.

Keep the token in Colab's secrets and never type it into a cell, otherwise anyone you share the notebook with can see it. The Qwen reader and the other models are not gated and work without a token, although downloads are faster with one.

The cell logs in to Hugging Face with the token. An earlier version of this notebook only placed the token in an environment variable, and loading Llama then failed with a `401 Unauthorized` error; logging in explicitly avoids that.

The cell also stops JAX and TensorFlow, which come pre-installed on Colab, from reserving GPU memory that PyTorch needs. Run it first every time, including after a reconnect or restart.

In [ ]:
import os, subprocess

# JAX and TensorFlow are pre-installed on Colab and would otherwise reserve most of the GPU.
os.environ["JAX_PLATFORMS"] = "cpu"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
os.environ["USE_TF"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Hugging Face token from Colab secrets, then an explicit login (needed for the gated Llama model).
token = None
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    pass
if token:
    os.environ["HF_TOKEN"] = token
    from huggingface_hub import login
    login(token=token, add_to_git_credential=False)
    print("Logged in to Hugging Face.")
else:
    print("No HF_TOKEN secret found. That is fine for the Qwen reader; Llama needs it (see the text above).")

try:
    out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.used,memory.total", "--format=csv,noheader,nounits"],
                         capture_output=True, text=True).stdout.strip()
    name, used, total = [x.strip() for x in out.split(",")]
    print(f"GPU: {name}, {(int(total) - int(used)) / 1024:.1f} GiB free of {int(total) / 1024:.1f} GiB")
    if int(used) > 0.2 * int(total):
        print("Something else is using the GPU. Choose Runtime > Restart session and run this cell again.")
except Exception:
    print("No GPU found. Choose Runtime > Change runtime type > T4 GPU, save, and run this cell again.")

## 2. Google Drive and the code

Colab will ask for permission to use your Google Drive. Everything is kept in the folder `My Drive/covid_rag_qa`, so a disconnect does not lose results. The full run needs about 2 GB of space there.

The cell then copies the code into that folder from the project repository on GitHub, whose address is already filled in as `REPO_URL`. If GitHub cannot be reached, upload the file `covid-rag-qa.zip` to the top level of your Drive and set `REPO_URL = ""`; the cell will use the zip instead.

Only the code folders (`src`, `configs`, `tests`, `scripts`) are replaced. Results and caches already in the folder are left as they are, so you can update the code without losing a finished run.

In [ ]:
REPO_URL = "https://github.com/Vidhvan10024/covid-rag-qa.git"   # leave empty to use covid-rag-qa.zip on your Drive instead

import os, shutil, subprocess, zipfile
from google.colab import drive
drive.mount("/content/drive")

PROJECT = "/content/drive/MyDrive/covid_rag_qa"
FRESH = "/content/covid-rag-qa"          # temporary copy of the repository, outside Drive
os.makedirs(PROJECT, exist_ok=True)
shutil.rmtree(FRESH, ignore_errors=True)

ZIP = "/content/drive/MyDrive/covid-rag-qa.zip"
if REPO_URL:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, FRESH], check=True)
elif os.path.exists(ZIP):
    with zipfile.ZipFile(ZIP) as z:
        z.extractall("/content")
    if not os.path.isdir(FRESH):   # the zip may hold the folder under another name
        top = [d for d in os.listdir("/content") if os.path.isfile(f"/content/{d}/run_all.sh")]
        shutil.move(f"/content/{top[0]}", FRESH)
else:
    raise SystemExit("Set REPO_URL above, or upload covid-rag-qa.zip to the top level of My Drive.")

for folder in ["src", "configs", "tests", "scripts"]:
    shutil.rmtree(os.path.join(PROJECT, folder), ignore_errors=True)
    shutil.copytree(os.path.join(FRESH, folder), os.path.join(PROJECT, folder))
for f in ["requirements.txt", "run_all.sh"]:
    shutil.copy(os.path.join(FRESH, f), PROJECT)

%cd /content/drive/MyDrive/covid_rag_qa
print("Code is in", PROJECT)

## 3. Python packages

Colab already has PyTorch, Transformers and Datasets. This adds BM25, the evaluation library and 4-bit loading, and takes about a minute. Red warnings about dependency conflicts can be ignored as long as the last line says *Packages ready*.

The versions are printed without importing the packages, because importing some of them at this point would take GPU memory.

In [ ]:
!pip -q install bm25s PyStemmer ranx bitsandbytes tabulate

from importlib.metadata import version
for pkg in ["torch", "transformers", "datasets", "bm25s", "ranx", "bitsandbytes"]:
    try:
        print(f"{pkg:13s} {version(pkg)}")
    except Exception:
        print(f"{pkg:13s} missing, run this cell again")
print("Packages ready")

## 4. Settings

Change only the first two lines.

`RUN_MODE = "pilot"` runs on 20,000 TREC-COVID documents and 200 questions. Use it first to see that everything works; its numbers are not the paper's. `RUN_MODE = "full"` is the complete experiment. Pilot and full results go to separate folders (`results_pilot`, `results_full`) and never mix.

`READER` is the model that writes the answers. Run the whole notebook with the primary reader, Qwen, first. For the second reader, set `READER = "meta-llama/Llama-3.1-8B-Instruct"`, keep `RUN_MODE = "full"`, run sections 1 to 4, and go straight to section 8. The Llama results go to `results_full_llama`, and it uses the passage setting that section 7 chose for Qwen, so only the reader changes.

Run this cell again after every reconnect.

In [ ]:
RUN_MODE = "full"                         # "pilot" or "full"
READER = "Qwen/Qwen2.5-7B-Instruct"        # or "meta-llama/Llama-3.1-8B-Instruct" for the second reader

import os, json, yaml
MAIN_READER = "Qwen/Qwen2.5-7B-Instruct"
cfg = yaml.safe_load(open("configs/default.yaml"))
cfg["output_dir"] = f"results_{RUN_MODE}"
cfg["cache_dir"] = "cache"
cfg["qa"]["reader_model"] = READER
cfg["trec_covid"]["max_docs"] = 20000 if RUN_MODE == "pilot" else None
cfg["qa"]["max_questions"] = 200 if RUN_MODE == "pilot" else None

if READER != MAIN_READER:
    cfg["output_dir"] = f"results_{RUN_MODE}_" + READER.split("/")[-1].split("-")[0].lower()
    best_path = f"results_{RUN_MODE}/exp3_ablation/best.json"
    if os.path.exists(best_path):
        best = json.load(open(best_path))
        cfg["covid_qa"]["chunk_words"], cfg["qa"]["top_k"] = best["chunk_words"], best["k"]
        print(f"Second reader: using the primary reader's setting, {best['chunk_words']}-word passages, k = {best['k']}.")
    else:
        print("The primary reader's section 7 result was not found. Finish the Qwen run first.")
    print("For the second reader, skip sections 5 to 7 and go to section 8.")

yaml.safe_dump(cfg, open("configs/run.yaml", "w"), sort_keys=False)
RESULTS = cfg["output_dir"]
print(f"Mode: {RUN_MODE}. Reader: {READER}. Results folder: {PROJECT}/{RESULTS}")

### Check progress

Shows what has finished and what is half done for the current settings. You can run it at any time when nothing else is running.

In [ ]:
import glob

def n_answers(path):
    try:
        return len(json.load(open(path))["outputs"])
    except Exception:
        return 0

print(f"{RESULTS}")
print("  Experiment 1 (retrieval):", "done" if os.path.exists(f"{RESULTS}/exp1_retrieval/table_retrieval.csv") else "not finished")
for f in sorted(glob.glob("cache/trec_emb/*.npy")):
    print("     saved embeddings:", os.path.basename(f))
for exp, label in [("exp3_ablation", "Experiment 3 (passage settings)"), ("exp2_main", "Experiment 2 (question answering)")]:
    print(f"  {label}:", "done" if os.path.exists(f"{RESULTS}/{exp}/summary.csv") else "not finished")
    for f in sorted(glob.glob(f"{RESULTS}/{exp}/generations/*.json*")):
        name = os.path.basename(f)
        if name.startswith("faith_"):
            continue
        state = "partial" if name.endswith(".partial") else "complete"
        print(f"     {name.replace('.json', '').replace('.partial', ''):40s} {state:8s} {n_answers(f)} answers")

## 5. Self-test

Runs the whole pipeline on ten toy documents with simple stand-in models, without downloading anything. It only checks that the code runs, so ignore the numbers. Continue when it prints *Self-test passed*.

In [ ]:
!python -W ignore -m src.run_retrieval --smoke > /tmp/smoke1.log 2>&1 && \
 python -W ignore -m src.run_qa --smoke --stage all > /tmp/smoke2.log 2>&1 && \
 python -m src.make_figures --root results_smoke > /tmp/smoke3.log 2>&1 && \
 echo "Self-test passed" || (echo "Self-test failed. Last lines of the log:"; tail -n 25 /tmp/smoke*.log)

## 6. Experiment 1: retrieval on TREC-COVID

Ranks the TREC-COVID collection with BM25, BGE and MedCPT, their fusions, and the cross-encoder re-ranker, and evaluates the rankings against the official relevance judgements. The first run downloads the collection (about 70 MB) and encodes every document with both dense models, which is the slow part. The embeddings are saved to Drive, so a rerun after a disconnect is much quicker.

At the end you should see three tables:

- **Retrieval results**, one row per system. Higher nDCG@10 is better, and a `*` marks a significant difference from BM25. `ndcg@10_judged_only` repeats the score with never-judged documents removed; a large gap between the two means a system returns papers that the 2020 assessors never saw.
- **Re-ranker check**: the share of judged document pairs that the re-ranker puts in the right order, next to its first stage. If the verdict says *SUSPECT*, the re-ranker is doing worse than chance and something is wrong with the setup. Otherwise a drop in nDCG@10 after re-ranking is a genuine result.
- **Hypotheses H1a to H1c**, with the outcome of each test.

For orientation, the full run in the paper gave nDCG@10 of 0.589 for BM25, 0.779 for BGE and 0.612 for MedCPT.

In [ ]:
!python -W ignore -m src.run_retrieval --config configs/run.yaml

## 7. Experiment 3: passage length and number of passages

Tries passages of 100, 200 and 400 words with k = 1, 3, 5 and 10 passages, using the hybrid retriever and the primary reader, on the development articles only. The test articles are not touched here. The setting with the highest F1 is written to `best.json` and used automatically in section 8. In the paper this was 100-word passages with k = 10.

The first run downloads the Qwen model (about 15 GB, 5 to 10 minutes). Answers are saved after each setting.

In [ ]:
!python -W ignore -m src.run_qa --config configs/run.yaml --stage ablation

## 8. Experiment 2: question answering

Asks the reader every test question under seven conditions:

| Condition | What the reader gets |
|---|---|
| `closed_book` | no passages; this is the lower bound |
| `bm25`, `bge`, `medcpt` | the top ten passages from each retriever |
| `hybrid` | the top ten after fusing the three retrievers |
| `hybrid_rerank` | the fused list re-ordered by the cross-encoder |
| `oracle` | only the passage that contains the answer; this is the upper bound for the reader |

This is the longest step. If Colab disconnects, reconnect, run sections 1 to 4, and run this cell again. Finished conditions are skipped and a half-finished one continues from its last save.

Two checks are printed at the end:

- `[sanity] PASS` or `FAIL` for the automatic attribution score. The oracle sees the right passage, so no other condition should score higher. If it fails, report attribution from the human labels only.
- A **transfer** table. For each retriever it compares F1 on the questions where the answer-bearing passage was retrieved with the oracle's F1 on the same questions. `equivalent (±0.05) = True` means the two are statistically equivalent within 0.05 F1.

In [ ]:
!python -W ignore -m src.run_qa --config configs/run.yaml --stage main

## 9. Tables and figures

Writes the summary tables (Markdown and LaTeX) and figures into the results folder and shows them here. When both readers have finished, the second part also draws Figs 2 to 4 exactly as they appear in the paper, into `paper_figures`.

In [ ]:
!python -m src.make_figures --root {RESULTS}

from IPython.display import Image, Markdown, display
for name in ["table1_retrieval.md", "table1b_reranker_check.md", "table2_qa.md", "table2b_significance.md",
             "table4_transfer.md", "table4b_transfer_vs_oracle.md", "table5_rank_agreement.md", "table3_ablation.md"]:
    p = os.path.join(RESULTS, name)
    if os.path.exists(p):
        display(Markdown(f"### {name}\n" + open(p).read()))

main, second = f"results_{RUN_MODE}", f"results_{RUN_MODE}_llama"
if all(os.path.exists(f"{d}/exp2_main/summary.csv") for d in [main, second]) and os.path.exists(f"{main}/exp3_ablation/summary.csv"):
    from src.paper_figures import make_all
    for p in make_all(main, second, "paper_figures"):
        display(Markdown(f"### {os.path.basename(p)}")); display(Image(p, width=700))
else:
    print("Figs 2 to 4 of the paper are drawn once both readers have finished section 8.")

## 10. Save the environment and download the results

Records the package versions and the GPU, which belong in the supplementary material, and downloads a zip of the results folder (without the per-condition answer files, which are large; `predictions.jsonl` contains all answers). Everything also stays in `My Drive/covid_rag_qa`.

In [ ]:
!pip freeze > {RESULTS}/environment.txt
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv > {RESULTS}/gpu.txt
!cp configs/run.yaml {RESULTS}/config_used.yaml
!cd {RESULTS} && zip -qr /content/{RESULTS}.zip . -x "*/generations/*"
from google.colab import files
files.download(f"/content/{RESULTS}.zip")

## If something goes wrong

**No GPU, or "Cannot connect to GPU backend".** Choose the T4 under *Runtime > Change runtime type*. If Colab refuses, the free GPU allowance is used up for now; try again after several hours, or use Kaggle (below).

**CUDA out of memory.** Restart the session, run sections 1 to 4, and rerun the step. Don't run another GPU notebook at the same time. While answering, the code already halves the batch size by itself when memory runs short.

**401 Unauthorized or "gated repo" when loading Llama.** Either the licence has not been accepted yet on the model page, or the token is missing. Check the `HF_TOKEN` secret, make sure *Notebook access* is on, and run section 1 again; it should print *Logged in to Hugging Face*.

**The page says "Resuming execution" but nothing happens.** The step has stopped. Reconnect, run sections 1 to 4, look at *Check progress*, and rerun the step.

**`ModuleNotFoundError: No module named 'src'`.** The notebook is not in the project folder. Run section 2 again.

**The TREC-COVID download fails.** The code falls back to the copy on Hugging Face. If that also fails, try again later.

**Starting one step from scratch.** Delete its folder (for example `results_full/exp2_main`) on Drive and run the step again.

**Kaggle instead of Colab.** Create a notebook, choose *GPU T4 x2* as accelerator, switch the internet on, and import this notebook. There is no Google Drive on Kaggle: in section 2 replace the Drive lines with `PROJECT = "/kaggle/working/covid_rag_qa"`, and download the results at the end of each session.

**After the full run.** Leave the code and settings as they are. The numbers to report are the ones produced with `config_used.yaml`.